# German Electricity Load - Forecasting Model Comparison (Version 4)

This notebook studies the German national electricity demand and forecasts it two years
ahead on a weekly grid. It runs the full pipeline: preparing and exploring the data,
testing for stationarity, fitting classical benchmarks, a **SARIMA** model and its
exogenous extension **SARIMAX**, a feature-based learner, and an hourly LSTM - then
compares them all on one 104-week hold-out.

**Distinctive choice for this version.** The feature-based stage is a **model bake-off**
between two families that the earlier versions did not use: **K-Nearest-Neighbours**
(instance-based - "weeks with similar conditions behave alike") and **Lasso** (an
L1-regularised linear model that performs automatic feature selection). Both are tuned
with time-series cross-validation and the better-validated one is carried forward as the
primary recursive forecaster, with the runner-up reported for context.

**Practices carried throughout**
- Differencing is justified from unit-root tests (AIC is only compared within one `d`).
- Every lag/rolling feature is backward-looking, so no future information leaks in.
- The machine-learning forecast is produced recursively (a true multi-step forecast) and,
  separately labelled, one-step-ahead for reference only.
- The hourly scaler is fitted on the training span alone; the open-loop LSTM is bounded so
  it cannot diverge; RMSE, MAE and MAPE are reported for every model.

## Part 0 - Setup and shared utilities

The first cell fixes the visual theme, a compact colour key, the reproducibility seed and
a single error-metric helper reused by every model. Error metrics are written out in NumPy
so the notebook does not depend on any external scoring utility.

In [ ]:
!pip install holidays --quiet
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---- fourth-version visual identity ----------------------------------------
plt.style.use('ggplot')
CLR = {
    'navy':    '#1b4965',
    'apricot': '#f4a259',
    'green':   '#5b8e7d',
    'red':     '#bc4b51',
    'gold':    '#e9c46a',
    'grey':    '#8d99ae',
}
plt.rc('axes', prop_cycle=plt.cycler(
    color=[CLR['navy'], CLR['apricot'], CLR['green'], CLR['red'], CLR['gold'], CLR['grey']]),
       facecolor='#fbfbfb')
plt.rc('figure', facecolor='white')
plt.rc('font', size=10)

RSTATE = 11
np.random.seed(RSTATE)


def errors(obs, pred):
    """Return RMSE, MAE and MAPE for one forecast as a dictionary."""
    obs, pred = np.asarray(obs, float), np.asarray(pred, float)
    gap = np.abs(obs - pred)
    return {
        'RMSE': float(np.sqrt(np.mean(gap ** 2))),
        'MAE': float(gap.mean()),
        'MAPE': float(np.mean(gap / np.abs(obs)) * 100.0),
    }


def unit_rmse(obs, pred):
    """Root-mean-square error only (used by the neural-network section)."""
    obs, pred = np.asarray(obs, float), np.asarray(pred, float)
    return float(np.sqrt(np.mean((obs - pred) ** 2)))

## Part 1 - Data preparation and exploratory analysis

We read the Open Power System Data 60-minute file for Germany, keep the actual-load column
from 2015 onward, and average it to daily and weekly series. The weekly series is the main
modelling grid; the raw hourly signal is reserved for the neural network in Part 6.

In [ ]:
import os

# Try the Kaggle location first, then a couple of local fall-backs, so the notebook
# also runs outside Kaggle. Edit DATA_LOCATIONS if your copy lives elsewhere.
DATA_LOCATIONS = [
    '/kaggle/input/datasets/rishiande/german/opsd_60min_raw.csv',
    'opsd_60min_raw.csv',
    'data/opsd_60min_raw.csv',
]
DATA_FILE = next((p for p in DATA_LOCATIONS if os.path.exists(p)), None)
if DATA_FILE is None:
    raise FileNotFoundError(
        'opsd_60min_raw.csv was not found. Point DATA_LOCATIONS at the OPSD 60-minute '
        'file (https://data.open-power-system-data.org/time_series/).')
panel = pd.read_csv(DATA_FILE, parse_dates=['utc_timestamp'], index_col='utc_timestamp')
print(f'Read {len(panel):,} hourly rows from {DATA_FILE}')

In [ ]:
LOAD_COL = 'DE_load_actual_entsoe_transparency'
grid_df = panel[[LOAD_COL]].rename(columns={LOAD_COL: 'mw'}).copy()
grid_df = grid_df.loc['2015-01-01':'2020-10-31'].dropna()
print('Span        :', grid_df.index.min(), '->', grid_df.index.max())
print('Hourly rows :', f'{len(grid_df):,}')

In [ ]:
hourly_load = grid_df['mw']
daily_load = hourly_load.resample('D').mean()
weekly_load = hourly_load.resample('W').mean()
print('Weekly points :', weekly_load.size, '| missing:', bool(weekly_load.isna().any()))
print(weekly_load.describe().round(1).to_string())

### Visual exploration

Two views: the daily series with a 12-week rolling mean to expose the trend, and a
**box-and-whisker plot of weekly load grouped by calendar month**, which summarises the
annual cycle and its spread far more compactly than a raw time plot.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 4.8))

ax1.plot(daily_load.index, daily_load, color=CLR['grey'], lw=0.6, alpha=0.8, label='Daily mean')
ax1.plot(daily_load.rolling(84, center=True).mean(), color=CLR['red'], lw=2.2, label='12-week rolling mean')
ax1.set_title('Daily electricity load')
ax1.set_ylabel('MW')
ax1.legend()

by_month = pd.DataFrame({'mw': weekly_load, 'month': weekly_load.index.month})
groups = [by_month.loc[by_month['month'] == m, 'mw'].to_numpy() for m in range(1, 13)]
bp = ax2.boxplot(groups, patch_artist=True, medianprops=dict(color=CLR['red']))
for box in bp['boxes']:
    box.set(facecolor=CLR['navy'], alpha=0.55)
ax2.set_xticklabels(['J', 'F', 'M', 'A', 'M', 'J', 'J', 'A', 'S', 'O', 'N', 'D'])
ax2.set_title('Weekly load distribution by month')
ax2.set_ylabel('MW')
ax2.set_xlabel('Month')
plt.tight_layout()
plt.show()

An additive decomposition (`period=52`) separates the trend, the annual seasonal wave and
the remainder. The seasonal amplitude is large relative to the trend drift, which is the
first sign that a plain "repeat last year" rule will be hard to beat.

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

parts = seasonal_decompose(weekly_load, model='additive', period=52)
fig, ax = plt.subplots(3, 1, figsize=(13.6, 6.6), sharex=True)
ax[0].plot(parts.observed.index, parts.observed, color=CLR['grey'], lw=1.0, label='Observed')
ax[0].plot(parts.trend.index, parts.trend, color=CLR['red'], lw=2.0, label='Trend')
ax[0].legend(loc='upper right', fontsize=8)
ax[0].set_ylabel('MW')
ax[1].plot(parts.seasonal.index, parts.seasonal, color=CLR['navy'], lw=1.0)
ax[1].set_ylabel('Seasonal')
ax[2].plot(parts.resid.index, parts.resid, color=CLR['green'], lw=0.9)
ax[2].axhline(0, color=CLR['grey'], lw=0.8)
ax[2].set_ylabel('Remainder')
ax[2].set_xlabel('Date')
ax[0].set_title('Additive decomposition of weekly load (period = 52)')
plt.tight_layout()
plt.show()

### Stationarity tests

The Augmented Dickey-Fuller (ADF) and KPSS tests examine stationarity from opposite null
hypotheses; the ACF/PACF plots then inform the differencing orders for the SARIMA model.

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf


def report_stationarity(col, tag):
    col = col.dropna()
    p_adf = adfuller(col)[1]
    p_kpss = kpss(col, regression='c', nlags='auto')[1]
    print(f'{tag:30s} ADF p={p_adf:6.4f} ({"stationary" if p_adf <= 0.05 else "unit root"})'
          f'  |  KPSS p={p_kpss:6.4f} ({"non-stationary" if p_kpss < 0.05 else "stationary"})')


print('Stationarity checks')
print('-' * 88)
report_stationarity(weekly_load, 'level')
report_stationarity(weekly_load.diff(), 'first difference')
report_stationarity(weekly_load.diff(52), 'seasonal difference (52)')

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(13.8, 7.2))
plot_acf(weekly_load.dropna(), ax=ax[0, 0], lags=104, title='ACF - level')
plot_pacf(weekly_load.dropna(), ax=ax[0, 1], lags=52, method='ywm', title='PACF - level')
plot_acf(weekly_load.diff().dropna(), ax=ax[1, 0], lags=104, title='ACF - first difference')
plot_pacf(weekly_load.diff().dropna(), ax=ax[1, 1], lags=52, method='ywm', title='PACF - first difference')
plt.tight_layout()
plt.show()

**Differencing choice.** ADF rejects a unit root at the level and KPSS does not flag
non-stationarity, so the level is close to mean-stationary; but the ACF decays slowly and
peaks near lag 52, confirming the annual cycle. The SARIMA therefore takes one seasonal
difference (`D = 1`, `s = 52`) to remove the yearly wave and one ordinary difference
(`d = 1`) for the residual drift. A second ordinary difference is unwarranted - the series
is already stationary after one - and the order search below confirms it numerically.

## Part 2 - Benchmark models

The final **104 weeks (two years)** are held out. Four classical forecasts are scored on
that window: the **Mean**, the **Naive** (last observed value carried forward), the
**Seasonal naive** (previous year repeated), and a **Drift** line.

In [ ]:
HORIZON = 104
SEASON_LEN = 52
train_w = weekly_load.iloc[:-HORIZON]
test_w = weekly_load.iloc[-HORIZON:]
test_idx = test_w.index
print(f'Train : {train_w.size} weeks (to {train_w.index[-1].date()})')
print(f'Test  : {test_w.size} weeks (to {test_w.index[-1].date()})')

In [ ]:
last_cycle = train_w.iloc[-SEASON_LEN:].to_numpy()
slope = (train_w.iloc[-1] - train_w.iloc[0]) / (train_w.size - 1)

benchmarks = {
    'Mean': pd.Series(train_w.mean(), index=test_idx),
    'Naive': pd.Series(train_w.iloc[-1], index=test_idx),
    'Seasonal naive': pd.Series(np.take(last_cycle, np.arange(HORIZON) % SEASON_LEN), index=test_idx),
    'Drift': pd.Series(train_w.iloc[-1] + slope * np.arange(1, HORIZON + 1), index=test_idx),
}
print('Benchmark scores on the hold-out')
for tag, series in benchmarks.items():
    e = errors(test_w, series)
    print(f"  {tag:16s} RMSE={e['RMSE']:8.1f}  MAE={e['MAE']:8.1f}  MAPE={e['MAPE']:5.2f}%")

In [ ]:
fig, ax = plt.subplots(figsize=(12.2, 4.6))
ax.plot(test_idx, test_w, color=CLR['navy'], lw=2.4, label='Actual (hold-out)')
styles = {'Seasonal naive': dict(color=CLR['red'], lw=1.8),
          'Mean': dict(color=CLR['green'], ls=(0, (4, 2))),
          'Drift': dict(color=CLR['apricot'], ls=(0, (1, 1))),
          'Naive': dict(color=CLR['grey'], ls=(0, (5, 1)))}
for tag, kw in styles.items():
    ax.plot(test_idx, benchmarks[tag], label=tag, **kw)
ax.set_title('Benchmark forecasts over the two-year hold-out')
ax.set_ylabel('MW')
ax.legend(ncol=3, fontsize=8)
plt.tight_layout()
plt.show()

## Part 3 - SARIMA

The brief requires a sweep over `p in [0,6]`, `d in [0,2]`, `q in [0,6]` (147 orders).
Because the likelihood - and hence AIC - is evaluated on the `d`-times-differenced series,
AIC is **not** comparable across different `d`. The procedure screens all orders, fixes
`d = 1` from the unit-root evidence, refits the leading `d = 1` orders exactly so their AIC
shares a basis, applies a parsimony tie-break, then searches a small seasonal grid.

In [ ]:
import itertools
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox
from joblib import Parallel, delayed

SEAS = 52
pdq_space = list(itertools.product(range(7), range(3), range(7)))
SEAS_INIT = (1, 1, 1, SEAS)
FLAGS = dict(enforce_stationarity=False, enforce_invertibility=False)
print('Non-seasonal orders to screen:', len(pdq_space))


def quick_aic(order, y):
    """Fast simple-differencing fit; the AIC only ranks orders that share one d."""
    try:
        res = SARIMAX(y, order=order, seasonal_order=SEAS_INIT, simple_differencing=True,
                      **FLAGS).fit(disp=False, method='lbfgs', maxiter=55)
        return {'d': order[1], 'order': order, 'aic': res.aic,
                'conv': bool((res.mle_retvals or {}).get('converged', False))}
    except Exception:
        return {'d': order[1], 'order': order, 'aic': np.inf, 'conv': False}


probe = pd.DataFrame(Parallel(n_jobs=-1)(delayed(quick_aic)(o, train_w) for o in pdq_space))
grid_low = probe.sort_values('aic').iloc[0]
print('Unconstrained grid minimum:', tuple(grid_low['order']),
      f"(AIC={grid_low['aic']:.2f}, d={int(grid_low['order'][1])}) - not comparable across d")

In [ ]:
D_STAR = 1


def resid_std(fitted, order, seasonal):
    """Standardized residuals with the state-space warm-up trimmed off."""
    warm = order[1] + seasonal[1] * seasonal[3]
    try:
        block = np.asarray(fitted.standardized_forecasts_error)
        block = block[0] if getattr(block, 'ndim', 1) > 1 else block
    except Exception:
        block = np.asarray(fitted.resid)
    return pd.Series(block).replace([np.inf, -np.inf], np.nan).iloc[warm:].dropna()


def full_fit(order, seasonal, y, exog=None):
    """Exact ML fit plus a Ljung-Box p-value on the whitened residuals."""
    res = SARIMAX(y, exog=exog, order=order, seasonal_order=seasonal,
                  **FLAGS).fit(disp=False, method='lbfgs', maxiter=360)
    wres = resid_std(res, order, seasonal)
    lb_k = max(1, min(18, len(wres) - 2))
    p = acorr_ljungbox(wres, lags=[lb_k], return_df=True)['lb_pvalue'].iloc[0]
    return res, bool((res.mle_retvals or {}).get('converged', False)), p


cands = (probe[(probe['d'] == D_STAR) & probe['conv']].sort_values('aic')
             .head(10)['order'].tolist())
if not cands:
    cands = probe[probe['d'] == D_STAR].sort_values('aic').head(10)['order'].tolist()
if not cands:
    cands = [(1, D_STAR, 1), (0, D_STAR, 1)]

recs = []
for order in cands:
    try:
        res, conv, p = full_fit(order, SEAS_INIT, train_w)
        recs.append({'order': order, 'aic': res.aic, 'bic': res.bic, 'conv': conv, 'LB_p': round(p, 4)})
    except Exception:
        recs.append({'order': order, 'aic': np.inf, 'bic': np.inf, 'conv': False, 'LB_p': np.nan})
exact_tbl = pd.DataFrame(recs)
exact_tbl = exact_tbl[np.isfinite(exact_tbl['aic'])].sort_values('aic', ignore_index=True)
if exact_tbl.empty:
    exact_tbl = pd.DataFrame([{'order': (1, D_STAR, 1), 'aic': np.nan, 'bic': np.nan, 'conv': False, 'LB_p': np.nan}])
print(f'Exact refits at d={D_STAR} (AIC now comparable):')
print(exact_tbl.to_string(index=False))

In [ ]:
# Parsimony (Burnham & Anderson): inside a 2-AIC band, keep the fewest AR+MA terms.
floor = float(exact_tbl['aic'].min())
tie = exact_tbl[exact_tbl['aic'] <= floor + 2.0]
if tie.empty:
    tie = exact_tbl.head(1)
order_pdq = min(tie['order'], key=lambda o: (o[0] + o[2], o[0]))
print('Best AIC order    :', exact_tbl['order'].iloc[0])
print('Within 2 AIC units:', list(tie['order']))
print('Parsimonious pick :', order_pdq)

### Seasonal order

With `s = 52` fixed by the annual cycle and `D = 1`, the seasonal `(P, Q)` terms are
searched over `{0, 1}`. The space is kept shallow on purpose: fewer than four full annual
cycles remain after a seasonal difference, so a richer seasonal grid would fit noise. The
lowest-AIC converged option is selected.

In [ ]:
seasonal_grid = [(P, 1, Q, SEAS) for P in (0, 1) for Q in (0, 1)]
seas_recs = []
for so in seasonal_grid:
    try:
        res, conv, p = full_fit(order_pdq, so, train_w)
        seas_recs.append({'seasonal': so, 'aic': res.aic, 'bic': res.bic, 'conv': conv, 'LB_p': round(p, 4)})
    except Exception:
        seas_recs.append({'seasonal': so, 'aic': np.inf, 'bic': np.inf, 'conv': False, 'LB_p': np.nan})
seas = pd.DataFrame(seas_recs)
seas = seas[np.isfinite(seas['aic'])].sort_values('aic', ignore_index=True)
if seas.empty:
    seas = pd.DataFrame([{'seasonal': SEAS_INIT, 'aic': np.nan, 'bic': np.nan, 'ok': False, 'LB_p': np.nan}])
print(seas.to_string(index=False))
order_seasonal = seas['seasonal'].iloc[0]
print('\nSelected seasonal order:', order_seasonal)

In [ ]:
import scipy.stats as sstat

sarima_model = SARIMAX(train_w, order=order_pdq, seasonal_order=order_seasonal,
                       **FLAGS).fit(disp=False, method='lbfgs', maxiter=450)

whitened = resid_std(sarima_model, order_pdq, order_seasonal)
p_norm = sstat.shapiro(whitened)[1]
lb_lags = [l for l in (10, 20, 52) if l < len(whitened)]
lb = acorr_ljungbox(whitened, lags=lb_lags, return_df=True)

bar = '=' * 58
print(bar)
print('FINAL SARIMA  {} x {}'.format(order_pdq, order_seasonal))
print(bar)
print('AIC / BIC          : {:.2f} / {:.2f}'.format(sarima_model.aic, sarima_model.bic))
print('Converged          :', bool(sarima_model.mle_retvals.get('converged', False)))
print('Shapiro-Wilk p     : {:.3f} ({})'.format(p_norm, 'normal' if p_norm > 0.05 else 'non-normal'))
print('Ljung-Box (whitened residuals):')
print(lb.to_string())

### Residual diagnostics

The whitened residuals are inspected with a correlogram and a histogram (drawn directly,
which avoids the diffuse-burn-in limitation of the built-in diagnostics helper on a short
seasonal series). Together with the Ljung-Box and Shapiro-Wilk figures above they tell us
whether the model has reduced the series to near-white noise.

In [ ]:
fig, (axa, axb) = plt.subplots(1, 2, figsize=(12.6, 4.2))
plot_acf(whitened, ax=axa, lags=min(52, len(whitened) // 2 - 1), title='Whitened residuals - ACF')
axb.hist(whitened, bins=24, color=CLR['navy'], alpha=0.7, edgecolor='white', density=True)
grid_x = np.linspace(float(whitened.min()), float(whitened.max()), 200)
axb.plot(grid_x, sstat.norm.pdf(grid_x), color=CLR['red'], lw=2, label='N(0, 1)')
axb.set_title('Whitened residuals - distribution')
axb.legend(fontsize=8)
plt.tight_layout()
plt.show()

Some autocorrelation usually survives at the longer lags and normality is rejected, so the
residuals are close to - but not exactly - white noise: holiday effects and the 2020 shock
are structure a purely seasonal model cannot encode. The 95% intervals are therefore
approximate, which is one reason we benchmark every model against the seasonal naive.

In [ ]:
sarima_bundle = sarima_model.get_forecast(steps=HORIZON)
sarima_fc = sarima_bundle.predicted_mean
sarima_fc.index = test_idx
sarima_ci = sarima_bundle.conf_int(alpha=0.05)
sarima_ci.index = test_idx

fig, ax = plt.subplots(figsize=(12.2, 4.6))
ax.plot(train_w.index[-75:], train_w.iloc[-75:], color=CLR['grey'], lw=1.1, label='Recent history')
ax.plot(test_idx, test_w, color=CLR['navy'], lw=2.2, label='Actual')
ax.plot(test_idx, sarima_fc, color=CLR['red'], lw=2.0, ls=(0, (5, 1)), label='SARIMA mean')
ax.fill_between(test_idx, sarima_ci.iloc[:, 0], sarima_ci.iloc[:, 1], color=CLR['red'], alpha=0.15,
                label='95% interval')
ax.set_title('SARIMA {} x {} forecast'.format(order_pdq, order_seasonal))
ax.set_ylabel('MW')
ax.legend(ncol=2, fontsize=8)
plt.tight_layout()
plt.show()

sarima_err = errors(test_w, sarima_fc)
print("SARIMA  RMSE={RMSE:.1f}  MAE={MAE:.1f}  MAPE={MAPE:.2f}%".format(**sarima_err))

## Part 4 - SARIMAX with temperature and holidays

Weekly mean temperature for Berlin (Open-Meteo archive) stands in for national weather. We
keep the level, its square (the heating/cooling response is U-shaped), a one-week lag, and a
German public-holiday flag. Because the hold-out uses *observed* future temperature, any
model that consumes it is a conditional (explanatory) forecast, not a truly operational one.

In [ ]:
import requests
import holidays

WEATHER_URL = 'https://archive-api.open-meteo.com/v1/archive'
WEATHER_ARGS = {'latitude': 52.52, 'longitude': 13.41, 'start_date': '2015-01-01',
                'end_date': '2020-09-30', 'hourly': 'temperature_2m', 'timezone': 'UTC'}
try:
    blob = requests.get(WEATHER_URL, params=WEATHER_ARGS, timeout=60).json()
except Exception as err:
    raise RuntimeError('Open-Meteo request failed - the SARIMAX and feature-based sections '
                       'need this temperature series. Check connectivity or cache it.') from err

t_series = pd.Series(blob['hourly']['temperature_2m'],
                        index=pd.to_datetime(blob['hourly']['time']))
t_series.index = t_series.index.tz_localize('UTC')
temp_w = t_series.resample('W').mean().reindex(weekly_load.index).interpolate().bfill().ffill()

de_days = holidays.Germany(years=range(2015, 2021))
holiday_w = pd.Series([int(any(d in de_days for d in pd.date_range(end=w, periods=7)))
                       for w in weekly_load.index], index=weekly_load.index)

drivers = pd.DataFrame({'temp': temp_w, 'temp_sq': temp_w ** 2,
                        'temp_lag': temp_w.shift(1).bfill(), 'holiday': holiday_w})
print('Drivers:', list(drivers.columns), '| nulls:', int(drivers.isna().sum().sum()))
drivers_train = drivers.iloc[:-HORIZON]
drivers_test = drivers.iloc[-HORIZON:]

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5.2))
sc = ax.scatter(drivers['temp'], weekly_load, c=weekly_load.index.month, cmap='twilight',
                s=22, alpha=0.85)
cbar = fig.colorbar(sc, ax=ax, ticks=[1, 4, 7, 10])
cbar.set_label('Month')
q = np.polyfit(drivers['temp'], weekly_load, 2)
xs = np.linspace(drivers['temp'].min(), drivers['temp'].max(), 120)
ax.plot(xs, np.polyval(q, xs), color=CLR['red'], lw=2.4, label='Quadratic fit')
ax.set_title('Weekly load vs temperature (coloured by month)')
ax.set_xlabel('Weekly mean temperature (C)')
ax.set_ylabel('Weekly mean load (MW)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
sarimax_model = SARIMAX(train_w, exog=drivers_train, order=order_pdq, seasonal_order=order_seasonal,
                        **FLAGS).fit(disp=False, method='lbfgs', maxiter=450)
sarimax_bundle = sarimax_model.get_forecast(steps=HORIZON, exog=drivers_test)
sarimax_fc = sarimax_bundle.predicted_mean
sarimax_fc.index = test_idx
sarimax_ci = sarimax_bundle.conf_int(alpha=0.05)
sarimax_ci.index = test_idx

sarimax_err = errors(test_w, sarimax_fc)
print("SARIMAX RMSE={RMSE:.1f}  MAE={MAE:.1f}  MAPE={MAPE:.2f}%".format(**sarimax_err))
print("SARIMA  RMSE={:.1f}  (no drivers)".format(sarima_err['RMSE']))

fig, ax = plt.subplots(figsize=(12.2, 4.6))
ax.plot(test_idx, test_w, color=CLR['navy'], lw=2.2, label='Actual')
ax.plot(test_idx, sarimax_fc, color=CLR['green'], lw=2.0, label='SARIMAX mean')
ax.fill_between(test_idx, sarimax_ci.iloc[:, 0], sarimax_ci.iloc[:, 1], color=CLR['green'], alpha=0.16,
                label='95% interval')
ax.set_title('SARIMAX (temperature + holiday) forecast')
ax.set_ylabel('MW')
ax.legend(ncol=2, fontsize=8)
plt.tight_layout()
plt.show()

Compare the SARIMAX RMSE with the plain SARIMA RMSE above. The drivers add explanatory
value, but they are only partly known ahead of time (temperature needs a forecast; holidays
are deterministic), so the figure is a conditional forecast.

## Part 5 - Feature-based model: KNN vs Lasso bake-off

The feature-based stage compares two model families that the earlier versions did not use:

- **K-Nearest-Neighbours (KNN)** - an instance-based learner that predicts a week from the
  average of the most similar past weeks, using scaled distance over the feature space. It
  captures non-linear "similar-conditions" structure but cannot extrapolate beyond the range
  of the training data.
- **Lasso** - an L1-regularised linear model that shrinks weak coefficients to exactly zero,
  giving automatic feature selection and an interpretable, low-variance fit.

Both need **scaled inputs** (KNN because it works on Euclidean distance, Lasso because the L1
penalty must treat features on a common scale), so each is wrapped in a `StandardScaler`
pipeline. The annual cycle is supplied as **Fourier terms** (two harmonics of the yearly
period) rather than a raw week number, which suits both a distance metric and a linear fit.
Both are tuned with `GridSearchCV` under a `TimeSeriesSplit`, and the better-validated model
is carried forward.

**Leakage control.** Every lag/rolling feature is backward-looking (`load_lag1 = shift(1)`,
`load_lag52 = shift(52)`, `temp_lag1 = shift(1)`, `temp_roll4 = trailing mean`), so no
feature at week *t* uses week *t* or later.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import Lasso
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.inspection import permutation_importance

woy = weekly_load.index.isocalendar().week.astype(int).to_numpy()
feat_tbl = pd.DataFrame(index=weekly_load.index)
feat_tbl['sin1'] = np.sin(2 * np.pi * woy / 52)
feat_tbl['cos1'] = np.cos(2 * np.pi * woy / 52)
feat_tbl['sin2'] = np.sin(4 * np.pi * woy / 52)
feat_tbl['cos2'] = np.cos(4 * np.pi * woy / 52)
feat_tbl['temp'] = drivers['temp']
feat_tbl['temp_sq'] = drivers['temp'] ** 2
feat_tbl['temp_lag1'] = drivers['temp'].shift(1)
feat_tbl['temp_roll4'] = drivers['temp'].rolling(4).mean()
feat_tbl['load_lag1'] = weekly_load.shift(1)
feat_tbl['load_lag52'] = weekly_load.shift(52)
feat_tbl['holiday'] = drivers['holiday']
feat_tbl['y'] = weekly_load.to_numpy()
feat_tbl = feat_tbl.dropna()

PRED_COLS = [c for c in feat_tbl.columns if c != 'y']
feat_train = feat_tbl.iloc[:-HORIZON]
feat_test = feat_tbl.iloc[-HORIZON:]
Xtr, ytr = feat_train[PRED_COLS], feat_train['y']
Xte, yte = feat_test[PRED_COLS], feat_test['y']
print('Feature matrix:', Xtr.shape, '| predictors:', PRED_COLS)

### Tuning and selection

Each family is grid-searched under an expanding-window time-series split (so validation folds
always lie in the future of their training folds). We report each family's best
cross-validated RMSE and chosen settings, then keep the winner.

In [ ]:
folds = TimeSeriesSplit(n_splits=4)
contenders = {
    'KNN': (Pipeline([('scale', StandardScaler()), ('model', KNeighborsRegressor())]),
            {'model__n_neighbors': [3, 5, 7, 10, 15], 'model__weights': ['uniform', 'distance']}),
    'Lasso': (Pipeline([('scale', StandardScaler()), ('model', Lasso(max_iter=50000))]),
              {'model__alpha': np.logspace(-1, 4, 14)}),
}

searched = {}
fam_rows = []
for name, (pipe, grid) in contenders.items():
    gs = GridSearchCV(pipe, grid, cv=folds, scoring='neg_root_mean_squared_error', n_jobs=-1)
    gs.fit(Xtr, ytr)
    searched[name] = gs
    fam_rows.append({'Family': name, 'CV RMSE': round(-gs.best_score_, 1),
                 'Best settings': {k.replace('model__', ''): v for k, v in gs.best_params_.items()}})
bakeoff = pd.DataFrame(fam_rows).sort_values('CV RMSE', ignore_index=True)
print(bakeoff.to_string(index=False))

best_name = bakeoff.iloc[0]['Family']
fb_model = searched[best_name].best_estimator_
print('\nBest-fit feature model:', best_name)

In [ ]:
# (a) one-step reference: each week uses the ACTUAL previous-week load.
fb_onestep = pd.Series(fb_model.predict(Xte), index=yte.index)

# (b) recursive multi-step from the training origin: the load lags are refilled with the
#     model's OWN predictions once past known history, so this matches SARIMA/SARIMAX.
#     Temperature and holiday stay observed (the conditional-forecast assumption).
timeline = weekly_load.index
base = train_w.size
history = list(weekly_load.iloc[:base].to_numpy())
temp_line = drivers['temp']
hol_line = drivers['holiday']

steps = []
for k in range(HORIZON):
    pos = base + k
    stamp = timeline[pos]
    wk = int(stamp.isocalendar()[1])
    row = {
        'sin1': np.sin(2 * np.pi * wk / 52), 'cos1': np.cos(2 * np.pi * wk / 52),
        'sin2': np.sin(4 * np.pi * wk / 52), 'cos2': np.cos(4 * np.pi * wk / 52),
        'temp': temp_line.iloc[pos], 'temp_sq': temp_line.iloc[pos] ** 2,
        'temp_lag1': temp_line.iloc[pos - 1], 'temp_roll4': temp_line.iloc[pos - 3:pos + 1].mean(),
        'load_lag1': history[pos - 1], 'load_lag52': history[pos - 52],
        'holiday': hol_line.iloc[pos],
    }
    pred = float(fb_model.predict(pd.DataFrame([row])[PRED_COLS])[0])
    steps.append(pred)
    history.append(pred)
fb_recursive = pd.Series(steps, index=test_idx)

one_err = errors(yte, fb_onestep)
rec_err = errors(test_w, fb_recursive)
print(f"{best_name} one-step   RMSE={one_err['RMSE']:8.1f}  MAE={one_err['MAE']:8.1f}  MAPE={one_err['MAPE']:5.2f}%  (actual lag-1)")
print(f"{best_name} recursive  RMSE={rec_err['RMSE']:8.1f}  MAE={rec_err['MAE']:8.1f}  MAPE={rec_err['MAPE']:5.2f}%  (true 2-yr)")

In [ ]:
fig, ax = plt.subplots(figsize=(12.2, 4.6))
ax.plot(feat_train.index[-75:], ytr.iloc[-75:], color=CLR['grey'], lw=1.1, label='Recent history')
ax.plot(test_idx, test_w, color=CLR['navy'], lw=2.2, label='Actual')
ax.plot(test_idx, fb_recursive, color=CLR['red'], lw=2.0, label=f'{best_name} recursive (multi-step)')
ax.plot(test_idx, fb_onestep, color=CLR['apricot'], lw=1.4, ls=(0, (2, 2)), label=f'{best_name} one-step (actual lag-1)')
ax.set_title(f'{best_name} forecasts')
ax.set_ylabel('MW')
ax.legend(ncol=2, fontsize=8)
plt.tight_layout()
plt.show()

### Which drivers matter

Neither KNN (no coefficients) nor a scaled Lasso pipeline exposes importance in a directly
comparable way, so we use **permutation importance** on the hold-out - shuffle one feature,
measure the rise in RMSE, repeat. A larger rise means the chosen model leaned on that feature
more. Shown with the mean and spread over repeats.

In [ ]:
perm = permutation_importance(fb_model, Xte, yte, n_repeats=25, random_state=RSTATE,
                              scoring='neg_root_mean_squared_error')
rank = perm.importances_mean.argsort()

fig, ax = plt.subplots(figsize=(9.2, 4.9))
ax.barh(np.array(PRED_COLS)[rank], perm.importances_mean[rank],
        xerr=perm.importances_std[rank], color=CLR['navy'], alpha=0.8,
        error_kw=dict(ecolor=CLR['grey']))
ax.set_title(f'{best_name} permutation importance (hold-out)')
ax.set_xlabel('Increase in RMSE when the feature is shuffled')
plt.tight_layout()
plt.show()

### Reading the bake-off

The winner is chosen purely on cross-validated RMSE, so the selection is honest and
reproducible (both learners are deterministic). The permutation ranking is typically led by
the recent-load and year-ago-load lags, echoing the strong annual persistence, with the
Fourier and temperature terms refining the fit. If **KNN** wins, the series rewards a local
"similar weeks" rule; if **Lasso** wins, a sparse linear combination of the engineered
features is enough - and Lasso's zeroed coefficients also tell us which drivers were
redundant. Either way the recursive multi-step forecast is the one comparable to SARIMA.

## Part 6 - Hourly LSTM

### Literature review

LSTM networks (Hochreiter and Schmidhuber, 1997) add gated memory to a recurrent cell so
gradients survive over long sequences, unlike plain RNNs. For electricity load, Kong et al.
(2017, *IEEE Transactions on Smart Grid*) show an LSTM learns short-run fluctuations and
seasonal shape directly from smart-meter data without the stationarity assumptions that
SARIMA models rely on.
Broader evidence is more measured: Hewamalage, Bergmeir and Bandara (2021, *International
Journal of Forecasting*) report that RNN/LSTM models are competitive but not automatically
better than statistical baselines and are sensitive to preprocessing, and Lim and Zohren
(2021, *Phil. Trans. R. Soc. A*) highlight compounding error in recursive multi-step
prediction as a recurring weakness. We model the **hourly** series with a 168-hour look-back
and quantify exactly those trade-offs below.

In [ ]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from sklearn.preprocessing import MinMaxScaler

tf.keras.utils.set_random_seed(RSTATE)

series_h = grid_df['mw'].to_numpy()
TEST_SPAN = 24 * 7 * 52 * 2               # last two years, in hours
train_span = len(series_h) - TEST_SPAN    # everything before the test window

# Scale on the TRAINING span only, then transform all of it - fitting on the full series
# would leak the test-window min/max into the training inputs.
scaler_mm = MinMaxScaler()
scaler_mm.fit(series_h[:train_span].reshape(-1, 1))
series_n = scaler_mm.transform(series_h.reshape(-1, 1)).ravel()

WINDOW = 168
X_seq = np.stack([series_n[i - WINDOW:i] for i in range(WINDOW, len(series_n))])[:, :, None]
y_seq = series_n[WINDOW:]
split = len(X_seq) - TEST_SPAN
X_fit, y_fit = X_seq[:split], y_seq[:split]
X_hold, y_hold = X_seq[split:], y_seq[split:]
print('LSTM tensors -> fit', X_fit.shape, '| hold-out', X_hold.shape)

### Architecture search and final model

Five recurrent designs are compared with early stopping, ranked by **validation loss** (the
proper selection signal, never the test set). For the **final** model we fix a two-layer
stacked LSTM with a linear output head: single-layer nets are prone to instability under long
recursion, and fixing the architecture keeps results reproducible across runs.

In [ ]:
def make_net(widths, drop):
    mdl = Sequential()
    mdl.add(Input(shape=(WINDOW, 1)))
    for j, w in enumerate(widths):
        mdl.add(LSTM(w, return_sequences=(j < len(widths) - 1)))
        mdl.add(Dropout(drop))
    mdl.add(Dense(1, activation='linear'))   # linear head - stable under recursion
    mdl.compile(optimizer='adam', loss='mse')
    return mdl


net_specs = [
    {'tag': 'single-32', 'widths': [32], 'drop': 0.15, 'batch': 128},
    {'tag': 'stack-64-32', 'widths': [64, 32], 'drop': 0.20, 'batch': 128},
    {'tag': 'stack-96-48', 'widths': [96, 48], 'drop': 0.30, 'batch': 64},
    {'tag': 'single-72', 'widths': [72], 'drop': 0.25, 'batch': 96},
    {'tag': 'deep-96-48-24', 'widths': [96, 48, 24], 'drop': 0.30, 'batch': 64},
]

log = []
for cfg in net_specs:
    tf.random.set_seed(RSTATE)
    rnn = make_net(cfg['widths'], cfg['drop'])
    early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    h = rnn.fit(X_fit, y_fit, epochs=10, batch_size=cfg['batch'], validation_split=0.1,
                callbacks=[early_stop], verbose=0)
    log.append({'design': cfg['tag'], 'widths': '-'.join(map(str, cfg['widths'])),
                'val_loss': round(min(h.history['val_loss']), 6)})
scan_tbl = pd.DataFrame(log).sort_values('val_loss', ignore_index=True)
print(scan_tbl.to_string(index=False))

# Fixed final architecture (reproducible; stable under recursion), not the per-run minimum.
final_design = {'tag': 'stack-96-48', 'widths': [96, 48], 'drop': 0.30, 'batch': 64}
print('\nValidation leader :', scan_tbl.iloc[0]['design'])
print('Final architecture:', final_design['tag'])

In [ ]:
tf.random.set_seed(RSTATE)
lstm = make_net(final_design['widths'], final_design['drop'])
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
history = lstm.fit(X_fit, y_fit, epochs=10, batch_size=final_design['batch'],
                   validation_split=0.1, callbacks=[early_stop], verbose=0)
print('Final network trained:', final_design['tag'])

In [ ]:
fig, ax = plt.subplots(figsize=(9.4, 4.2))
ax.plot(history.history['loss'], color=CLR['navy'], marker='o', ms=3, label='Train')
ax.plot(history.history['val_loss'], color=CLR['red'], marker='s', ms=3, label='Validation')
ax.set_title('LSTM training curve')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE (scaled)')
ax.legend()
plt.tight_layout()
plt.show()

### Rolling versus open-loop

A **rolling** one-step forecast is fed the actual previous 168 hours at each step; an
**open-loop** forecast feeds its own predictions back, the only honest way to cover two years
from a single origin. Each fed-back value is clipped to the trained `[0, 1]` range so the
recursion cannot drift to implausible values. Both are reported and are not interchangeable.

In [ ]:
# rolling one-step: uses the genuine previous 168 hours each step
roll = scaler_mm.inverse_transform(lstm.predict(X_hold, batch_size=256, verbose=0)).ravel()
truth = scaler_mm.inverse_transform(y_hold.reshape(-1, 1)).ravel()

# open-loop recursion with the feedback clipped to the trained range
buf = series_n[split:split + WINDOW].copy()
loop = []
for step in range(TEST_SPAN):
    nxt = lstm.predict(buf.reshape(1, WINDOW, 1), verbose=0)[0, 0]
    nxt = min(1.0, max(0.0, float(nxt)))   # bound the recursion to the trained range
    loop.append(nxt)
    buf = np.concatenate([buf[1:], [nxt]])
    if (step + 1) % 2500 == 0:
        print(f'   open-loop {step + 1}/{TEST_SPAN}')
loop = scaler_mm.inverse_transform(np.array(loop).reshape(-1, 1)).ravel()

hidx = grid_df.index[WINDOW + split:]
roll_wk = pd.Series(roll, index=hidx).resample('W').mean()
truth_wk = pd.Series(truth, index=hidx).resample('W').mean()
loop_wk = pd.Series(loop, index=hidx[:TEST_SPAN]).resample('W').mean()
loop_truth_wk = pd.Series(truth[:TEST_SPAN], index=hidx[:TEST_SPAN]).resample('W').mean()

print(f'Rolling   RMSE hourly : {unit_rmse(truth, roll):8.1f} MW  [one-step, sees actuals]')
print(f'Rolling   RMSE weekly : {unit_rmse(truth_wk, roll_wk):8.1f} MW  [one-step, sees actuals]')
print(f'Open-loop RMSE weekly : {unit_rmse(loop_truth_wk, loop_wk):8.1f} MW  [true multi-step]')
print(f'Open-loop RMSE hourly : {unit_rmse(truth[:TEST_SPAN], loop):8.1f} MW  [true multi-step]')

The rolling weekly RMSE sits below the hourly one because weekly averaging cancels
independent hourly errors - but it is a one-step forecast reading the true recent past, so it
must not be compared with the multi-step models. Even bounded by the clip, the open-loop
weekly RMSE stays well above the seasonal-naive benchmark: recursive feedback compounds error
over two years, so a univariate open-loop LSTM is not viable at this horizon without external
drivers. Both weekly LSTM figures come from resampling the hourly test window, so predicted
and actual align to each other; the weekly bins differ marginally from the weekly-model index,
a granularity nuance rather than a leak.

## Part 7 - Answers to the assignment questions

**Q1 - Which models meaningfully beat the seasonal-naive benchmark?**
Comparisons are only fair within a forecast type. On multi-step weekly RMSE the recursive
feature-based model is the closest challenger to the seasonal naive - at best drawing level
with it within run-to-run noise - while SARIMAX, SARIMA and the remaining naive baselines
trail it and the open-loop LSTM is far adrift through recursive drift. No multi-step model
improves on "repeat last year" by a margin large enough to be decisive: German weekly demand
is dominated by a stable annual cycle, and the hold-out contains the 2020 COVID-19 dip - a
shock none of the models anticipate but the seasonal naive sidesteps by reusing the pre-shock
profile. The one-step feature model and the rolling LSTM look sharper only because they read
the actual previous value, and must not be compared with the multi-step baseline.

**Q2 - How was data leakage avoided in the temperature features?**
Every engineered feature is backward-looking: `temp_lag1 = shift(1)`, `temp_roll4` is a
trailing four-week mean, `load_lag1 = shift(1)` and `load_lag52 = shift(52)`. No feature at
week *t* uses week *t* or later. Current-week temperature and the holiday flag are used only
under the explicit conditional-forecast assumption, and the recursive forecast refills its own
load lags with predictions past the origin, so no future actual leaks into the multi-step
result. The hourly scaler is fitted on the training span alone for the same reason.

**Q3 - Justify the differencing orders and seasonal period.**
`d = 1`: ADF/KPSS agree the level is essentially stationary after one difference and `d = 2`
over-differences. `D = 1`, `s = 52`: the decomposition and ACF show a dominant annual cycle in
weekly data, removed by one seasonal difference at lag 52. The seasonal `(P, Q)` were searched
over `{0, 1}`, and the ordinary `(p, q)` were chosen by within-`d` AIC plus a parsimony rule.

**Q4 - Do the covariates help, and are they known at the origin?**
Temperature (with a squared term for the U-shaped response), its lag and the holiday flag
move the SARIMAX RMSE relative to plain SARIMA. They are only partly known ahead of time -
temperature needs a weather forecast, holidays are deterministic - so the result is
conditional rather than operational.

**Q5 - Interpretability and complexity.**

| Aspect | SARIMAX | KNN / Lasso | LSTM |
|---|---|---|---|
| Interpretability | High (coefficients, intervals) | KNN low; Lasso high (sparse coefficients) | Low (black box) |
| Complexity | Low-medium | Low (few hyper-parameters) | High (architecture + GPU) |
| Uncertainty | Native intervals | None natively | None natively |
| Non-linearity | Manual (squared term) | KNN native; Lasso linear only | Native (learned) |
| Training cost | Minutes | Seconds | Minutes to hours |

**Q6 - Which model for operational use?**
We recommend **SARIMAX**. It is not the most accurate here (the seasonal naive and the
recursive feature model match or beat it), but it uniquely combines native prediction
intervals for risk planning, interpretable temperature/holiday coefficients for scenario
analysis, direct support for exogenous drivers, and cheap retraining. The feature-based model
is a strong, low-cost competitor but has no native uncertainty; the open-loop LSTM is
unsuitable over this long horizon. A deployment should keep tracking the seasonal-naive
benchmark and retrain as new, post-shock data arrives.

## Part 8 - Consolidated comparison

Every model is gathered with a forecast-type tag so like is compared with like. The sorted
bar chart shows weekly RMSE against the seasonal-naive line, and the overlay shows the
multi-step forecasts against the hold-out with the open-loop LSTM in its own panel.

In [ ]:
def summary_row(tag, actual, predicted, kind):
    e = errors(actual, predicted)
    return {'Model': tag, 'Type': kind, 'RMSE [MW]': round(e['RMSE'], 1),
            'MAE [MW]': round(e['MAE'], 1), 'MAPE [%]': round(e['MAPE'], 2)}


fb_tag = f'{best_name} (feature-based)'
table = pd.DataFrame([
    summary_row('Mean', test_w, benchmarks['Mean'], 'multi-step'),
    summary_row('Naive', test_w, benchmarks['Naive'], 'multi-step'),
    summary_row('Seasonal naive', test_w, benchmarks['Seasonal naive'], 'multi-step'),
    summary_row('Drift', test_w, benchmarks['Drift'], 'multi-step'),
    summary_row('SARIMA', test_w, sarima_fc, 'multi-step'),
    summary_row('SARIMAX', test_w, sarimax_fc, 'multi-step (conditional)'),
    summary_row(f'{best_name} recursive', test_w, fb_recursive, 'multi-step (conditional)'),
    summary_row(f'{best_name} one-step', yte, fb_onestep, 'one-step (actual lag-1)'),
    summary_row('LSTM open-loop', loop_truth_wk, loop_wk, 'multi-step'),
    summary_row('LSTM rolling', truth_wk, roll_wk, 'one-step (actual lag-1)'),
]).sort_values('RMSE [MW]', ignore_index=True)
base = table.loc[table['Model'] == 'Seasonal naive', 'RMSE [MW]'].iloc[0]
table['vs seasonal naive'] = (table['RMSE [MW]'] - base).round(1)
print('Seasonal-naive weekly RMSE =', base, 'MW\n')
print(table.to_string(index=False))

In [ ]:
ms = table[table['Type'].str.startswith('multi-step')].sort_values('RMSE [MW]', ascending=True)
colours = [CLR['red'] if m == ms['RMSE [MW]'].min() else CLR['navy'] for m in ms['RMSE [MW]']]

fig, ax = plt.subplots(figsize=(11.2, 5.4))
ax.barh(ms['Model'], ms['RMSE [MW]'], color=colours, alpha=0.85)
ax.axvline(base, color=CLR['grey'], ls='--', lw=1.3, label=f'Seasonal naive ({base:.0f} MW)')
for y, v in zip(range(len(ms)), ms['RMSE [MW]']):
    ax.text(v, y, f' {v:.0f}', va='center', fontsize=8)
ax.invert_yaxis()
ax.set_title('Multi-step weekly RMSE (lower is better)')
ax.set_xlabel('RMSE [MW]')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(15.6, 6.2))
ax.plot(train_w.index, train_w, color=CLR['grey'], lw=0.8, alpha=0.7, label='Train')
ax.plot(test_idx, test_w, color=CLR['navy'], lw=2.4, label='Actual (hold-out)')
ax.plot(test_idx, benchmarks['Seasonal naive'], color=CLR['gold'], lw=1.4, ls=(0, (5, 2)), label='Seasonal naive')
ax.plot(test_idx, sarima_fc, color=CLR['green'], lw=1.4, label='SARIMA')
ax.plot(test_idx, sarimax_fc, color=CLR['apricot'], lw=1.4, label='SARIMAX')
ax.plot(test_idx, fb_recursive, color=CLR['red'], lw=2.2, label=f'{best_name} recursive')
lo = min(train_w.min(), test_w.min()) * 0.9
hi = max(train_w.max(), test_w.max()) * 1.1
ax.set_ylim(lo, hi)
ax.set_title('Multi-step forecasts vs actual (open-loop LSTM shown separately)')
ax.set_ylabel('MW')
ax.legend(ncol=3, fontsize=8, loc='lower left')
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(15.6, 3.6))
ax.plot(test_idx, test_w, color=CLR['navy'], lw=2.2, label='Actual (hold-out)')
ax.plot(loop_wk.index, loop_wk, color=CLR['red'], lw=1.6, label='LSTM open-loop (recursive)')
ax.set_title('Open-loop LSTM over the two-year horizon (bounded by clipping)')
ax.set_ylabel('MW')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()